# Guidelines for Prompting
In this lesson, you'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.

## Setup
#### Load the API key and relevant Python libaries if needed.

In [1]:
%pip install -qU openai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip show openai

Name: openai
Version: 2.8.1
Summary: The official Python library for the openai API
Home-page: 
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /media/limcheekin/My Passport/ws/py/prompt-engineering-for-developers/.venv/lib/python3.12/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: langchain-openai
Note: you may need to restart the kernel to use updated packages.


__OpenAI API helper function__

Throughout this course, we will use LiquidAI's `lfm2-8B-A1B` model and the [chat completions endpoint](https://platform.openai.com/docs/guides/chat) of llama.cpp's llama-server.

This helper function will make it easier to use prompts and look at the generated outputs:

In [9]:
from openai import OpenAI, omit

BASE_URL = "http://192.168.1.111:8886/v1"
MODEL = "lfm2-8B-A1B"
DEFAULT_SYSTEM_PROMPT = "You are a helpful assistant, you will complete the task by follow the instructions given."

client = OpenAI(base_url=BASE_URL, api_key="sk-1")

def get_completion(prompt, tools=[], model=MODEL):
    messages = [
        { "role": "system", "content": DEFAULT_SYSTEM_PROMPT },
        { "role": "user", "content": prompt }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools = tools,
        temperature=0, 
        max_tokens=1024
    )

    msg = response.choices[0].message
    if msg.content:
        return msg.content
    elif msg.tool_calls:
        return msg.tool_calls
    else:
        return None


In [3]:
get_completion("What is AI?")

'AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think, learn, and make decisions. These systems can perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, solving problems, making predictions, and even creating content. AI encompasses various subfields, including machine learning (where systems improve from experience), deep learning (a subset using neural networks), natural language processing (understanding and generating human language), computer vision (interpreting visual data), and robotics. AI is used in many applications today, from virtual assistants and recommendation systems to autonomous vehicles and medical diagnostics.'

## Prompting Principles
- **Principle 1: Write clear and specific instructions**
- **Principle 2: Give the model time to “think”**

### Tactics

#### Tactic 1: Use delimiters to clearly indicate distinct parts of the input
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`

In [5]:
text = f"""
You should express what you want a model to do by
providing instructions that are as clear and
specific as you can possibly make them.
This will guide the model towards the desired output, 
and reduce the chances of receiving irrelevant
or incorrect responses. Don't confuse writing a 
clear prompt with writing a short prompt.
In many cases, longer prompts provide more clarity 
and context for the model, which can lead to
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks 
into a single sentence:.
```{text}```
"""
response = get_completion(prompt)
print(response)

To ensure accurate and relevant model outputs, provide clear, specific instructions that are sufficiently detailed—longer prompts often yield better results by offering necessary context and reducing ambiguity.


#### Tactic 2: Ask for a structured output
- https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat
- JSON, HTML

In [21]:
from pydantic import BaseModel
from typing import List

class Book(BaseModel):
    book_id: int
    title: str
    author: str
    genre: str

class BookList(BaseModel):
    books: list[Book]

prompt = f"""
Generate a list of three made-up book titles along
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""

messages = [
    { "role": "system", "content": DEFAULT_SYSTEM_PROMPT },
    { "role": "user", "content": prompt },
]

response = client.chat.completions.parse(
    model=MODEL,
    messages=messages,
    temperature=0, 
    max_tokens=1024,
    response_format=BookList
)

bookList = response.choices[0].message.parsed

print(len(bookList.books))
print(bookList.books[0])
print(bookList.books[1])

3
book_id=1 title='The Clockwork Heart of Elaria' author='Lira Voss' genre='Fantasy'
book_id=2 title='Echoes Beneath the Neon Sky' author='Kai Ren' genre='Science Fiction'


#### Tactic 3: Ask the model to check whether conditions are satisfied

In [23]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some 
water boiling. While that's happening, 
grab a cup and put a tea bag in it. Once the water is 
hot enough, just pour it over the tea bag. 
Let it sit for a bit so the tea can steep. After a  
few minutes, take out the tea bag. If you 
like, you can add some sugar or milk to taste. 
And that's it! You've got yourself a delicious 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Boil water until it is hot enough.  
Step 2 - Grab a cup and place a tea bag inside.  
Step 3 - Pour the hot water over the tea bag.  
Step 4 - Let the tea steep for a few minutes.  
Step 5 - Remove the tea bag after steeping.  
Step 6 - Add sugar or milk if desired.  
Step 7 - Enjoy your delicious cup of tea.


In [24]:
text_2 = f"""
The sun is shining brightly today, and the birds are 
singing. It's a beautiful day to go for a 
walk in the park. The flowers are blooming, and the 
trees are swaying gently in the breeze. People 
are out and about, enjoying the lovely weather. 
Some are having picnics, while others are playing 
games or simply relaxing on the grass. It's a 
perfect day to spend time outdoors and appreciate the 
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


#### Tactic 4: "Few-shot" prompting

In [25]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest 
valley flows from a modest spring; the 
grandest symphony originates from a single note; 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

Resilience is like the willow tree bending in a storm—strong yet flexible, never breaking. It is the quiet strength to rise again after falling, to endure hardship without losing hope. Like a river that carves through rock over time, resilience grows not in one mighty leap but in small, steady acts of perseverance. It means learning from setbacks, adapting with grace, and trusting that even in darkness, light returns.


### Principle 2: Give the model time to “think” 

#### Tactic 1: Specify the steps required to complete a task

In [26]:
text = f"""
In a charming village, siblings Jack and Jill set out on 
a quest to fetch water from a hilltop 
well. As they climbed, singing joyfully, misfortune 
struck—Jack tripped on a stone and tumbled 
down the hill, with Jill following suit. 
Though slightly battered, the pair returned home to 
comforting embraces. Despite the mishap, 
their adventurous spirits remained undimmed, and they 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple 
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following 
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1 - The siblings Jack and Jill embark on a joyful but accident-prone journey to fetch water, facing a fall but returning home with unbroken spirits.  
2 - Les frères Jack et Jill entreprennent une aventure joyeuse mais malheureuse pour chercher de l’eau, chutent ensemble mais reviennent chez eux avec des esprits intacts.  
3 - Jack, Jill  
4  
```  
{
  "french_summary": "Les frères Jack et Jill entreprennent une aventure joyeuse mais malheureuse pour chercher de l’eau, chutent ensemble mais reviennent chez eux avec des esprits intacts.",
  "num_names": 2
}


#### Ask for output in a specified format

In [27]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in Italian summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Text:  
In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well, but Jack tripped and fell while climbing, followed by Jill, though both returned home safely with their spirits undimmed.  

Summary:  
Les frères Jack et Jill entreprennent une aventure pour chercher de l’eau au sommet d’une colline, mais Jack trébuche et tombe, tandis que Jill le suit, bien qu’ils rentrent sains et saufs.  

Translation:  
Les frères Jack et Jill entreprennent une aventure pour chercher de l’eau au sommet d’une colline, mais Jack trébuche et tombe, tandis que Jill le suit, bien qu’ils rentrent sains et saufs.  

Names:  
Jack, Jill  

Output JSON:  
{"french_summary": "Les frères Jack et Jill entreprennent une aventure pour chercher de l’eau au sommet d’une colline, mais Jack trébuche et tombe, tandis que Jill le suit, bien qu’ils rentrent sains et saufs.", "num_names": 2}


#### Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion

In [12]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \ 
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

<>:22: SyntaxWarning: invalid escape sequence '\ '
<>:22: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/1265626465.py:22: SyntaxWarning: invalid escape sequence '\ '
  """


The student's solution is correct. The total cost for the first year of operations as a function of the number of square feet is indeed 450x + 100,000.


#### Note that the student's solution is actually not correct.
#### We can fix this by instructing the model to work out its own solution first.

In [13]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem. 
- Then compare your solution to the student's solution \ 
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

<>:56: SyntaxWarning: invalid escape sequence '\ '
<>:56: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/1880077714.py:56: SyntaxWarning: invalid escape sequence '\ '
  """


Question:
```
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: $100/square foot * x square feet = $100x
2. Solar panel cost: $250/square foot * x square feet = $250x
3. Maintenance cost: Flat $100,000 + $10/square foot * x square feet = $100,000 + $10x
Total cost: $100x + $250x + $100,000 + $10x = $360x + $100,000
```
Is the student's 

## Model Limitations: Hallucinations
- Boie is a real company, the product name is not real.

In [14]:
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

The AeroGlide UltraSlim Smart Toothbrush by Boie is an advanced, high-tech toothbrush designed to provide an efficient and effective oral hygiene experience. Here are some key features and benefits of this smart toothbrush:

1. Smart Technology: The toothbrush is equipped with Bluetooth connectivity, allowing it to sync with the Boie app on your smartphone. This enables you to track your brushing habits, receive real-time feedback, and set personalized brushing goals.

2. Pressure Sensor: The toothbrush features a built-in pressure sensor that alerts you if you're brushing too hard, helping to prevent gum damage and enamel wear.

3. Multiple Brushing Modes: The UltraSlim Smart Toothbrush offers several brushing modes, including clean, white, gum care, and sensitive modes. Each mode is designed to target specific oral care needs, ensuring a comprehensive cleaning experience.

4. 2-Minute Timer: The toothbrush includes a built-in 2-minute timer that ensures you brush for the recommended 

## Try experimenting on your own!

#### A note about the backslash
- In the course, we are using a backslash `\` to make the text fit on the screen without inserting newline '\n' characters.
- GPT-3 isn't really affected whether you insert newline characters or not.  But when working with LLMs in general, you may consider whether newline characters in your prompt may affect the model's performance.

# Iterative Prompt Development
In this lesson, you'll iteratively analyze and refine your prompts to generate marketing copy from a product fact sheet.

## Generate a marketing product description from a product fact sheet

In [15]:
fact_sheet_chair = """
OVERVIEW
- Part of a beautiful family of mid-century inspired office furniture, 
including filing cabinets, desks, bookcases, meeting tables, and more.
- Several options of shell color and base finishes.
- Available with plastic back and front upholstery (SWC-100) 
or full upholstery (SWC-110) in 10 fabric and 6 leather options.
- Base finish options are: stainless steel, matte black, 
gloss white, or chrome.
- Chair is available with or without armrests.
- Suitable for home or business settings.
- Qualified for contract use.

CONSTRUCTION
- 5-wheel plastic coated aluminum base.
- Pneumatic chair adjust for easy raise/lower action.

DIMENSIONS
- WIDTH 53 CM | 20.87”
- DEPTH 51 CM | 20.08”
- HEIGHT 80 CM | 31.50”
- SEAT HEIGHT 44 CM | 17.32”
- SEAT DEPTH 41 CM | 16.14”

OPTIONS
- Soft or hard-floor caster options.
- Two choices of seat foam densities: 
 medium (1.8 lb/ft3) or high (2.8 lb/ft3)
- Armless or 8 position PU armrests 

MATERIALS
SHELL BASE GLIDER
- Cast Aluminum with modified nylon PA6/PA66 coating.
- Shell thickness: 10 mm.
SEAT
- HD36 foam

COUNTRY OF ORIGIN
- Italy
"""

In [16]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)

response

"Introducing the Mid-Century Inspired Office Chair, a perfect blend of style and functionality for your home or business setting. This chair is part of a beautiful family of mid-century inspired office furniture, including filing cabinets, desks, bookcases, meeting tables, and more, all designed to complement each other and create a cohesive look in your workspace.\n\nCrafted with durability and comfort in mind, this chair features a 5-wheel plastic coated aluminum base, ensuring smooth and stable movement. The ergonomic design includes a pneumatic chair adjust, allowing for easy raise/lower action to suit your preferences. With a choice of shell colors and base finishes, including stainless steel, matte black, gloss white, or chrome, you can customize the chair to match your existing décor.\n\nThe SWC-100 model comes with plastic back and front upholstery, while the SWC-110 model offers full upholstery in 10 fabric and 6 leather options, providing a wide range of styles to choose from

## Issue 1: The text is too long 
- Limit the number of words/sentences/characters.

In [17]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)

response

'Introducing our mid-century inspired office chair, part of a stunning furniture collection. Choose from various shell colors and base finishes, including stainless steel, matte black, gloss white, or chrome. Available with plastic back/front upholstery (SWC-100) or full upholstery (SWC-110) in 10 fabric and 6 leather options. Features a 5-wheel plastic coated aluminum base, pneumatic chair adjust, and adjustable seat height. Suitable for home or business settings, qualified for contract use. Made in Italy with HD36 foam seat and cast aluminum shell.'

In [18]:
len(response.split(" "))

81

## Issue 2. Text focuses on the wrong details
- Ask it to focus on the aspects that are relevant to the intended audience.

In [19]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)

response

'Introducing our mid-century inspired office chair, crafted from durable cast aluminum with a modified nylon coating, featuring a 10mm shell thickness. Available in various shell colors and base finishes, with optional full upholstery in 10 fabric and 6 leather options. Equipped with a pneumatic adjust and 5-wheel base for optimal comfort and functionality in both home and business settings.'

In [20]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)

response

"Introducing the SWC-100 or SWC-110, a mid-century inspired office chair crafted from a 10mm cast aluminum shell with a modified nylon coating. Featuring a 5-wheel plastic coated aluminum base and pneumatic adjustability, this chair offers a choice of shell colors and base finishes, including stainless steel, matte black, gloss white, or chrome. Available with or without armrests, it's suitable for home or business settings, qualified for contract use. \n\nProduct IDs: SWC-100, SWC-110"

## Issue 3. Description needs a table of dimensions
- Ask it to extract information and organize it in a table.

In [21]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

After the description, include a table that gives the 
product's dimensions. The table should have two columns.
In the first column include the name of the dimension. 
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website. 
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_chair}```
"""

response = get_completion(prompt)

response

'<div>\n<h2>Mid-Century Inspired Office Chair</h2>\n<p>This elegant office chair is part of a beautiful family of mid-century inspired office furniture, including filing cabinets, desks, bookcases, meeting tables, and more. The chair is available in several options of shell color and base finishes, with plastic back and front upholstery (SWC-100) or full upholstery (SWC-110) in 10 fabric and 6 leather options. The base finish options are stainless steel, matte black, gloss white, or chrome. The chair is available with or without armrests, making it suitable for both home and business settings. It is qualified for contract use.</p>\n<p>Construction:</p>\n<ul>\n<li>5-wheel plastic coated aluminum base</li>\n<li>Pneumatic chair adjust for easy raise/lower action</li>\n</ul>\n<p>Dimensions:</p>\n<table>\n  <tr>\n    <th>Dimension</th>\n    <th>Measurement (inches)</th>\n  </tr>\n  <tr>\n    <td>WIDTH</td>\n    <td>20.87"</td>\n  </tr>\n  <tr>\n    <td>DEPTH</td>\n    <td>20.08"</td>\n  </t

## Load Python libraries to view HTML

In [22]:
from IPython.display import display, HTML
display(HTML(response))

Dimension,Measurement (inches)
WIDTH,"20.87"""
DEPTH,"20.08"""
HEIGHT,"31.50"""
SEAT HEIGHT,"17.32"""
SEAT DEPTH,"16.14"""


## Try experimenting on your own!

# Summarizing
In this lesson, you will summarize text with a focus on specific topics.

## Text to summarize

In [23]:
prod_review = """
Got this panda plush toy for my daughter's birthday, \
who loves it and takes it everywhere. It's soft and \ 
super cute, and its face has a friendly look. It's \ 
a bit small for what I paid though. I think there \ 
might be other options that are bigger for the \ 
same price. It arrived a day earlier than expected, \ 
so I got to play with it myself before I gave it \ 
to her.
"""

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/2733959966.py:1: SyntaxWarning: invalid escape sequence '\ '
  prod_review = """


## Summarize with a word/sentence/character limit

In [24]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)

Panda plush toy received positive feedback for its softness, cuteness, and friendly face. However, it was considered small for the price. Arrived early, allowing the reviewer to enjoy it before gifting.


## Summarize with a focus on shipping and delivery

In [25]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
Shipping deparmtment. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects \
that mention shipping and delivery of the product. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The panda plush toy was cute and soft, but a bit small for the price. It arrived early, but I found other options larger for the same cost.


## Summarize with a focus on price and value

In [26]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
pricing deparmtment, responsible for determining the \
price of the product.  

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects \
that are relevant to the price and perceived value. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The panda plush toy is soft and cute, but too small for the price. It arrived early, but other options might offer better value.


#### Comment
- Summaries include topics that are not related to the topic of focus.

## Try "extract" instead of "summarize"

In [27]:
prompt = f"""
Your task is to extract relevant information from \ 
a product review from an ecommerce site to give \
feedback to the Shipping department. 

From the review below, delimited by triple quotes \
extract the information relevant to shipping and \ 
delivery. Limit to 30 words. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)

<>:11: SyntaxWarning: invalid escape sequence '\ '
<>:11: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/2053972773.py:11: SyntaxWarning: invalid escape sequence '\ '
  """


The panda plush toy arrived a day earlier than expected. It's soft and cute but a bit small for the price. Other bigger options might be available for the same price.


## Summarize multiple product reviews

In [28]:

review_1 = prod_review 

# review for a standing lamp
review_2 = """
Needed a nice lamp for my bedroom, and this one \
had additional storage and not too high of a price \
point. Got it fast - arrived in 2 days. The string \
to the lamp broke during the transit and the company \
happily sent over a new one. Came within a few days \
as well. It was easy to put together. Then I had a \
missing part, so I contacted their support and they \
very quickly got me the missing piece! Seems to me \
to be a great company that cares about their customers \
and products. 
"""

# review for an electric toothbrush
review_3 = """
My dental hygienist recommended an electric toothbrush, \
which is why I got this. The battery life seems to be \
pretty impressive so far. After initial charging and \
leaving the charger plugged in for the first week to \
condition the battery, I've unplugged the charger and \
been using it for twice daily brushing for the last \
3 weeks all on the same charge. But the toothbrush head \
is too small. I’ve seen baby toothbrushes bigger than \
this one. I wish the head was bigger with different \
length bristles to get between teeth better because \
this one doesn’t.  Overall if you can get this one \
around the $50 mark, it's a good deal. The manufactuer's \
replacements heads are pretty expensive, but you can \
get generic ones that're more reasonably priced. This \
toothbrush makes me feel like I've been to the dentist \
every day. My teeth feel sparkly clean! 
"""

# review for a blender
review_4 = """
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \ 
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \ 
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \ 
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

reviews = [review_1, review_2, review_3, review_4]

<>:38: SyntaxWarning: invalid escape sequence '\ '
<>:38: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/58617423.py:38: SyntaxWarning: invalid escape sequence '\ '
  review_4 = """


In [29]:
for i in range(len(reviews)):
    prompt = f"""
    Your task is to generate a short summary of a product \ 
    review from an ecommerce site. 

    Summarize the review below, delimited by triple \
    backticks in at most 20 words. 

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")


<>:10: SyntaxWarning: invalid escape sequence '\ '
<>:10: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/3928838135.py:10: SyntaxWarning: invalid escape sequence '\ '
  """


0 The panda plush toy is soft, cute, and arrived early, but it's small for the price. Daughter loves it and takes it everywhere. 

1 The lamp arrived quickly, had extra storage, and the company promptly replaced broken parts and missing pieces, showing great customer service. 

2 Electric toothbrush with good battery life, but small head and expensive replacement heads. Overall, feels like a dentist visit daily. 

3 Product review: 17-piece system on sale for $49 in November, price increased to $70-$89 by December. Base quality decreased, motor failed after a year. 



## Try experimenting on your own!

# Inferring
In this lesson, you will infer sentiment and topics from product reviews and news articles.

## Product review text

In [30]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [31]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the given product review is positive. The reviewer expresses satisfaction with the lamp's features, fast delivery, and the company's responsive customer support. They mention a minor issue with a broken string during transit, but the company quickly resolved it. Overall, the review reflects a positive experience with Lumina and their products.


In [32]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

positive


## Identify types of emotions

In [33]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

- happiness
- satisfaction
- gratitude
- appreciation
- delight


## Identify anger

In [34]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [35]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{"Item": "lamp", "Brand": "Lumina"}
****


## Doing multiple tasks at once

In [36]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{"Sentiment": "positive", "Anger": false, "Item": "lamp", "Brand": "Lumina"}
```


## Inferring topics

In [37]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

## Infer 5 topics

In [38]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

1. Survey
2. NASA
3. Satisfaction
4. John Smith
5. Social Security Administration


In [39]:
response.split(sep=',')

['1. Survey\n2. NASA\n3. Satisfaction\n4. John Smith\n5. Social Security Administration']

In [40]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

## Make a news alert for certain topics

In [41]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as list with 0 or 1 for each topic.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

[1, 0, 0, 1, 0]


In [42]:
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep='\n')}
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

IndexError: list index out of range

## Try experimenting on your own!

# Transforming

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.


## Translation

LLM is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.


In [43]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)


<>:4: SyntaxWarning: invalid escape sequence '\ '
<>:4: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/1126866978.py:4: SyntaxWarning: invalid escape sequence '\ '
  """


Hola, me gustaría pedir una licuadora.


In [44]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)


The language of the sentence "Combien coûte le lampadaire?" is French.


In [45]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)


Here are the translations:

English: "I want to order a basketball"

French: "Je veux commander une balle de basket"

Spanish: "Quiero pedir una pelota de baloncesto"


In [46]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)


Formal: ¿Le gustaría pedir un cojín?

Informal: ¿Te gustaría pedir un cojín?


### Universal Translator

Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!


In [47]:
user_messages = [
    # System performance is slower than normal
    "La performance du système est plus lente que d'habitude.",
    # My monitor has pixels that are not lighting
    "Mi monitor tiene píxeles que no se iluminan.",
    # My mouse is not working
    "Il mio mouse non funziona",
    # My keyboard has a broken control key
    "Mój klawisz Ctrl jest zepsuty",
    "我的屏幕在闪烁" # My screen is flashing
]


In [48]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")


Original message (The language of the given text is French. The sentence translates to "The system performance is slower than usual." in English.): La performance du système est plus lente que d'habitude.
English Translation: The performance of the system is slower than usual.

Korean Translation: 시스템의 성능은 일반보다 느리다. 

Original message (The language of the given text is Spanish. The sentence translates to "My monitor has pixels that don't light up." in English.): Mi monitor tiene píxeles que no se iluminan.
English Translation: My monitor has pixels that don't light up.

Korean Translation: 내 모니터에는 켜지지 않는 픽셀이 있습니다. 

Original message (The language of the phrase "Il mio mouse non funziona" is Italian. It translates to "My mouse doesn't work" in English.): Il mio mouse non funziona
Here is the translation of the text "Il mio mouse non funziona" in English and Korean:

English: My mouse is not working.

Korean: 나의 마우스가 작동하지 않아요. (Nae-ui maueseoga jagdonghaji anayo.) 

Original message (The

## Try it yourself!

Try some translations on your own!


## Tone Transformation

Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [49]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)


Dear Sir/Madam,

I hope this message finds you well. I am writing to bring to your attention a particular specification of a standing lamp that I believe would be of interest to you.

Best regards,
Joe


## Format Conversion

ChatGPT can translate between formats. The prompt should describe the input and output formats.


In [50]:
data_json = {"resturant employees": [
    {"name": "Shyam", "email": "shyamjaiswal@gmail.com"},
    {"name": "Bob", "email": "bob32@gmail.com"},
    {"name": "Jai", "email": "jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)


Here is the code to convert the given JSON dictionary to an HTML table:

```python
import json

# Given JSON dictionary
json_dict = '{"resturant employees": [{"name": "Shyam", "email": "shyamjaiswal@gmail.com"}, {"name": "Bob", "email": "bob32@gmail.com"}, {"name": "Jai", "email": "jai87@gmail.com"}]}'

# Parse the JSON dictionary
data = json.loads(json_dict)

# Extract the table headers
headers = list(data['resturant employees'][0].keys())

# Create the HTML table
html_table = "<table>\n"
html_table += "<tr>" + "".join([f"<th>{header}</th>" for header in headers]) + "</tr>\n"
html_table += "".join([f"<tr>" + "".join([f"<td>{value}</td>" for value in row.values()]) + "</tr>\n" for row in data['resturant employees']])
html_table += "</table>"

print(html_table)
```

```python
import unittest

class TestHTMLTable(unittest.TestCase):
    def test_html_table(self):
        expected_html = "<table>\n<tr><th>name</th><th>email</th></tr>\n<tr><td>Shyam</td><td>shyamjaiswal@gmail.com</td></tr>

In [51]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response.

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.


In [52]:
text = [
    # The girl has a ball.
    "The girl with the black and white puppies have a ball.",
    "Yolanda has her notebook.",  # ok
    "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
    "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
    "Your going to need you’re notebook.",  # Homonyms
    "That medicine effects my ability to sleep. Have you heard of the butterfly affect?",  # Homonyms
    "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)


The girl with the black and white puppies has a ball.
No errors found.
Its going to be a long day. Does the car need its oil changed?
Their goes my freedom. There going to bring they're suitcases.
Here is the corrected version:

```
You're going to need your notebook.
```
That medicine effects my ability to sleep Have you heard of the butterfly affect
This phrase is to check chatGPT for spelling ability


In [53]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)


Here is the corrected version of the review:

```
I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it, though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.
```


In [54]:
%pip install redlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 1.6 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 8.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [55]:
from redlines import Redlines

diff = Redlines(text, response)
display(Markdown(diff.output_markdown))


<span style='color:red;font-weight:700;text-decoration:line-through;'>Got </span><span style='color:green;font-weight:700;'>Here is the corrected version of the review: </span>

<span style='color:green;font-weight:700;'>``` </span>

<span style='color:green;font-weight:700;'>I got </span>this for my daughter for her birthday <span style='color:red;font-weight:700;text-decoration:line-through;'>cuz </span><span style='color:green;font-weight:700;'>because </span>she keeps taking mine from my room.  Yes, adults also like pandas too.  She takes it everywhere with her, and it's super soft and cute.  One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for <span style='color:red;font-weight:700;text-decoration:line-through;'>it </span><span style='color:green;font-weight:700;'>it, </span>though. I think there might be other options that are bigger for the same price.  It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.<span style='color:green;font-weight:700;'></span>

<span style='color:green;font-weight:700;'>```</span>

In [56]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

Here is the proofread and corrected review in markdown format, targeting an advanced reader and following the APA style guide:

```markdown
## Review of the Panda Plush Toy

The panda plush toy, purchased for my daughter's birthday, exemplifies the enduring appeal of this beloved animal among adults. Despite its intended use as a child's toy, the plush's quality and charm have made it a cherished item for both the giver and the recipient. The toy's softness and cuteness are undeniable, making it an ideal companion for daily activities.

However, the plush does have a minor flaw: one of its ears is slightly lower than the other, which may not have been an intentional design choice. This asymmetry detracts slightly from the overall aesthetic appeal. Additionally, the size of the toy is somewhat smaller than anticipated, given the price point. It is worth considering whether larger alternatives at the same price could offer better value.

The delivery was prompt, arriving a day earlier than expected, which allowed me to enjoy the toy before gifting it to my daughter. This timely arrival added to the overall positive experience of purchasing this item.

### References
American Psychological Association. (2020). *Publication manual of the American Psychological Association* (7th ed.). Washington, DC: Author.
```


## Try it yourself!
Try changing the instructions to form your own review.

Thanks to the following sites:

https://writingprompts.com/bad-grammar-examples/


# Expanding
In this lesson, you will generate customer service emails that are tailored to each customer's review.

## Customize the automated reply to a customer email

In [57]:
# given the sentiment from the lesson on "inferring",
# and the original customer message, customize the email
sentiment = "negative"

# review for a blender
review = f"""
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \ 
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \ 
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \ 
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

<>:37: SyntaxWarning: invalid escape sequence '\ '
<>:37: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_65555/2049866193.py:37: SyntaxWarning: invalid escape sequence '\ '
  """


In [58]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt)
print(response)

Subject: Thank you for your review - We're sorry to hear about your experience

Dear Valued Customer,

Thank you for taking the time to share your review with us. We appreciate your feedback and the time you've invested in providing us with your thoughts on our products.

We understand that you had a negative experience with our seasonal sale pricing and the quality of the product. We apologize for any inconvenience this may have caused you. We strive to provide our customers with the best value and quality, and we regret that we fell short in this instance.

Please know that your feedback is valuable to us, and we will take your concerns into consideration as we continue to improve our products and services. If you have any further questions or concerns, please don't hesitate to reach out to our customer service team. They are available to assist you and address any issues you may have.

Once again, thank you for your review, and we hope to have the opportunity to serve you better in 

## Use Temperature 0.1

In [59]:
def get_completion(prompt, model="gpt-3.5-turbo", temperature=0.0):
    messages = [{"role": "user", "content": get_prompt(instruction=prompt)}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
        max_tokens=1000,
    )
    return response.choices[0].message["content"]    

In [60]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt, temperature=0.1)
print(response)

NameError: name 'get_prompt' is not defined

## Temparature

In [ ]:
response = get_completion("my favorite food is", temperature=0.1)
print(response)

In [ ]:
response = get_completion("my favorite food is", temperature=1.0)
print(response)

In [ ]:
response = get_completion("my favorite food is", temperature=2.0)
print(response)

## Try experimenting on your own!

# The Chat Format

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

In [ ]:

def format(messages=[], system="", user="Human: ", assistant="AI: "):
    """
    Format the messages from the API into human readable strings.
    """
    formatted_message = ""

    for message in messages:
        if message['role'] == 'system':
            formatted_message += f"{system}{message['content']}\n"
        elif message['role'] == 'user':
            formatted_message += f"{user}{message['content']}\n"
        elif message['role'] == 'assistant':
            formatted_message += f"{assistant}{message['content']}\n"

    return formatted_message


def get_completion_from_messages(messages=[], temperature=0.0):
    prompt = format(messages)
    return get_completion(prompt, temperature=temperature)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'} ]

print(format(messages))

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  
]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [ ]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [ ]:
import panel as pn  # GUI
pn.extension()


panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=1)
print(response)

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

## Overview
The notebook is created to tackle the problem of incomplete generation of sentence due to max_tokens settings.

Initial thought on solving the problem with the following ways:
1. Use stop sequences
2. Increase max_tokens
3. Use prompt to restrict the length/tokens of the generated text

### Use stop sequences

In [ ]:
response = get_completion("What is AI?", params={"stop": ["\n"]})
print(response)

### Increase max_tokens

In [ ]:
response = get_completion("What is AI?", params={"max_tokens": 512})
print(response)


### Use prompt to restrict the length/tokens of the generated text

In [ ]:
response = get_completion("What is AI? Limit the answer to 128 characters.", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("What is AI? Limit the answer to 128 tokens.", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("What is AI? Limit the answer to 60 words.", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("What is AI? Limit the answer to 60 words and make sure it is end in full sentence.", 
                          params={
                                "max_tokens": 128,
                                "stop": ["\n"]
                              })
print(response)

In [ ]:
response = get_completion("Limit the answer to 128 characters. What is AI?", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("Limit the answer to 128 tokens. What is AI?", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("Limit the answer to 60 words. What is AI?", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("Limit the answer to 60 words and make sure it is a complete sentence. What is AI?", params={"max_tokens": 128})
print(response)

In [ ]:
response = get_completion("Make sure the answer is end in full sentence. What is AI?", 
                          params={"max_tokens": 64})
print(response)

In [ ]:
response = get_completion("Answer the question. What is AI?", 
                          params={"max_tokens": 128})
print(response)

### Conclusion
Stop sequences is the preferred way to make sure the generated text most likely end in complete sentence. You can learn more about stop sequences at https://help.openai.com/en/articles/5072263-how-do-i-use-stop-sequences.

Otherwise, if the generated text is end in mid-sentence, the user can refresh the answer and the app to send the generated incomplete sentence to the API in order to continue generating the sentence completely.